In [56]:
"""
Add more as needed
"""
from transformer_lens import HookedTransformer
import torch as t
from torch import Tensor
import einops



In [55]:
"""
Save just a little more time when adding more libraries
"""
gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

/var/folders/lx/h48548hj3c1d6b032lzbhqz00000gn/T/ipykernel_61548/2061229548.py:4: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  gpt2_small: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


### Inspecting the Model

In [7]:
prompt = "Mechanistic Interpretability (MI) is the study of reverse engineering neural networks. " \
"Taking an inscrutable stack of matrices where we know that it works, and trying to reverse engineer how it works." \
"And often this inscrutable stack of matrices can be decompiled to a human interpretable algorithm! " \
"In my (highly biased) opinion, this is one of the most exciting research areas in ML."

loss = gpt2_small(prompt, return_type="loss")
logits = gpt2_small(prompt, return_type="logits")
both = gpt2_small(prompt, return_type="both")

print("Model Loss", loss)
print("Logits shape", logits.shape)
print("Both", both)

print(gpt2_small.cfg)

Model Loss tensor(3.6181, grad_fn=<DivBackward0>)
Logits shape torch.Size([1, 84, 50257])
Both Output(logits=tensor([[[ 7.5261, 11.1214,  7.8919,  ..., -3.1299, -3.3873,  8.5934],
         [ 4.3875,  4.9281,  3.2611,  ..., -3.5831, -0.3450,  4.9199],
         [ 6.4270,  6.8365,  4.1351,  ..., -5.7150, -1.6948,  6.0407],
         ...,
         [ 4.2531,  4.8188,  0.6015,  ..., -2.1171, -2.9114,  2.7283],
         [12.2532,  7.2645,  6.5724,  ..., -1.8752, -1.5544,  7.6423],
         [ 6.5671,  5.9576,  5.7340,  ..., -3.6372, -6.2290, 13.0225]]],
       grad_fn=<ViewBackward0>), loss=tensor(3.6181, grad_fn=<DivBackward0>))
HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(8.0),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': 

### Converting from Tokens to Strings

In [ ]:
print(gpt2_small.to_str_tokens("Keep"))
print(gpt2_small.to_str_tokens(["tokenize", "more", "strings"]))
print(gpt2_small.to_tokens(["tokenize", "more", "strings"]))
print(gpt2_small.to_string(t.tensor([30001, 1096])))
print(gpt2_small.to_string([30001, 1096]))


"""
We'll now determine how many tokens the model guessed correctly
We're quite litrally checking what the model 
Don't be shy to extensively use the ".shape" 
"""

logits: Tensor = gpt2_small(prompt, return_type="logits")
# Here the shape is (batch, posn, d_vocab)
# So for each position we have d_vocab probabilities
# print(logits.softmax(-1)[0, 83].sum())
prediction = logits.argmax(-1).squeeze()[:-1]
print(logits.shape)
true_tokens = gpt2_small.to_tokens(prompt).squeeze()[1:]
correct = (prediction == true_tokens).sum()

print("The number of correctly predicted tokens are: ", correct.item() , "/" , (true_tokens.shape[0]))


['<|endoftext|>', 'Keep']
[['<|endoftext|>', 'token', 'ize'], ['<|endoftext|>', 'more'], ['<|endoftext|>', 'strings']]
tensor([[50256, 30001,  1096],
        [50256,  3549, 50256],
        [50256, 37336, 50256]])
tokenize
tokenize
tensor(1.0000, grad_fn=<SumBackward0>)
torch.Size([1, 84, 50257])
The number of correctly predicted tokens are:  29 / 83


### Caching all Activations

In [ ]:
gpt2_tokens = gpt2_small.to_tokens(prompt)
gpt2_logits, gpt2_cache = gpt2_small.run_with_cache(gpt2_tokens, remove_batch_dim=True)


print(type(gpt2_logits), type(gpt2_cache))
#print(gpt2_cache)
#print(gpt2_cache['blocks.0.hook_attn_out'])

layer0_pattern_from_cache = gpt2_cache["pattern", 0]
l0_hook_q = gpt2_cache['blocks.0.attn.hook_q']
#print(l0_hook_q.shape)
l0_hook_k = gpt2_cache['blocks.0.attn.hook_k']
#print(l0_hook_k.shape)
_, _, d_head = l0_hook_k.shape
attn_scores_l0 = einops.einsum(l0_hook_q, l0_hook_k, "posQ n_heads d_head, posK n_heads d_head -> n_heads posQ posK")
n_heads, posQ, posK = attn_scores_l0.shape

attn_scores_scaled = attn_scores_l0 / (d_head)**0.5
mask = t.tril(t.ones(posQ, posK))
attn_scores_scaled_masked = attn_scores_scaled.masked_fill(mask==0, -1e9)
attn_scored_probs_l0 = attn_scores_scaled_masked.softmax(-1)


#print(layer0_pattern_from_cache - attn_scored_probs_l0)



<class 'torch.Tensor'> <class 'transformer_lens.ActivationCache.ActivationCache'>
ActivationCache with keys ['hook_embed', 'hook_pos_embed', 'blocks.0.hook_resid_pre', 'blocks.0.ln1.hook_scale', 'blocks.0.ln1.hook_normalized', 'blocks.0.attn.hook_q', 'blocks.0.attn.hook_k', 'blocks.0.attn.hook_v', 'blocks.0.attn.hook_attn_scores', 'blocks.0.attn.hook_pattern', 'blocks.0.attn.hook_z', 'blocks.0.hook_attn_out', 'blocks.0.hook_resid_mid', 'blocks.0.ln2.hook_scale', 'blocks.0.ln2.hook_normalized', 'blocks.0.mlp.hook_pre', 'blocks.0.mlp.hook_post', 'blocks.0.hook_mlp_out', 'blocks.0.hook_resid_post', 'blocks.1.hook_resid_pre', 'blocks.1.ln1.hook_scale', 'blocks.1.ln1.hook_normalized', 'blocks.1.attn.hook_q', 'blocks.1.attn.hook_k', 'blocks.1.attn.hook_v', 'blocks.1.attn.hook_attn_scores', 'blocks.1.attn.hook_pattern', 'blocks.1.attn.hook_z', 'blocks.1.hook_attn_out', 'blocks.1.hook_resid_mid', 'blocks.1.ln2.hook_scale', 'blocks.1.ln2.hook_normalized', 'blocks.1.mlp.hook_pre', 'blocks.1.mlp.